# ine


In [ ]:
from pyspark.sql import functions as F, DataFrame
from pyspark.sql.types import (
    IntegerType, LongType, BooleanType, DateType, StringType, TimestampType,
)

PAIS_ISO3  = "GTM"
CIE10_RE   = r"^[A-Z][0-9]{2}[0-9A-Z]?$"
_SENTINELS = ("Desconocido", "No Especificado", "Ignorado")

# ── DIM_ETARIO grid canónico (docs/modelo_dimensional.md §5.2) ────────────────
# (id_etario, grupo_edad_codigo, grupo_edad_nombre, edad_min, edad_max, categoria_etaria)
_DIM_ETARIO_GRID = [
    (1,  "LT1",   "Menores de 1",      0, 0,    "Niñez"),
    (2,  "01-04", "1 a 4",             1, 4,    "Niñez"),
    (3,  "05-09", "5 a 9",             5, 9,    "Niñez"),
    (4,  "10-14", "10 a 14",          10, 14,   "Niñez"),
    (5,  "15-19", "15 a 19",          15, 19,   "Juventud"),
    (6,  "20-24", "20 a 24",          20, 24,   "Juventud"),
    (7,  "25-29", "25 a 29",          25, 29,   "Juventud"),
    (8,  "30-34", "30 a 34",          30, 34,   "Adultez"),
    (9,  "35-39", "35 a 39",          35, 39,   "Adultez"),
    (10, "40-44", "40 a 44",          40, 44,   "Adultez"),
    (11, "45-49", "45 a 49",          45, 49,   "Adultez"),
    (12, "50-54", "50 a 54",          50, 54,   "Adultez"),
    (13, "55-59", "55 a 59",          55, 59,   "Adultez"),
    (14, "60-64", "60 a 64",          60, 64,   "Adultez"),
    (15, "65-69", "65 a 69",          65, 69,   "Vejez"),
    (16, "70-74", "70 a 74",          70, 74,   "Vejez"),
    (17, "75-79", "75 a 79",          75, 79,   "Vejez"),
    (18, "80-84", "80 a 84",          80, 84,   "Vejez"),
    (19, "85+",   "85 y más",         85, None, "Vejez"),
    (98, "UNK",   "No especificada",  None, None, "No especificado"),
    (99, "ALL",   "Todas las edades", 0, None, "Total"),
]
# id_etario -> categoria_etaria (para roll-up Niñez/Juventud/Adultez/Vejez/Total)
_ETARIO_CAT_MAP = F.create_map(
    *[x for row in _DIM_ETARIO_GRID for x in (F.lit(row[0]), F.lit(row[5]))]
)

# ── Normalización de causa a GBD Nivel 2 (docs/modelo_dimensional.md §6) ───────
# INE trae CIE-10 fino (A09, J18…). Se mapea cada código a su padre GBD Nivel 2
# vía los rangos ICD-10 del master GBD (Deaths.XLSX / Appendix Table 6, 16 causas).
# Reglas: COVID (U07) es causa propia CG0995 aunque el master lo solape; un código
# de 3 chars con >1 dueño GBD -> NULL (no se fuerza). Ver scripts/transformation/
# build_gbd_crosswalk.py (misma lógica, genera el CSV de referencia).
MAPEO_GBD = {
    "Neoplasms":                                    ("CG0600", 410),
    "Cardiovascular diseases":                      ("CG0530", 491),
    "Neglected tropical diseases and malaria":      ("CG0250", 344),
    "Nutritional deficiencies":                     ("CG0470", 386),
    "Diabetes and kidney diseases":                 ("CG0510", 955),
    "Mental disorders":                             ("CG0610", 545),
    "Neurological disorders":                       ("CG0590", 542),
    "Self-harm and interpersonal violence":         ("CG0860", 717),
    "COVID-19":                                     ("CG0995", 1013),
    "Transport injuries":                           ("CG0810", 688),
    "Digestive diseases":                           ("CG0580", 526),
    "Unintentional injuries":                       ("CG0850", 696),
    "Diarrheal diseases":                           ("CG0350", 302),
    "HIV/AIDS and sexually transmitted infections": ("CG0290", 366),
    "Respiratory infections and tuberculosis":      ("CG0390", 337),
    "Chronic respiratory diseases":                 ("CG0570", 508),
}
GBD_L2_RANGES = {
    "Neoplasms": "C00-C13.9, C15-C22.8, C23-C25.9, C30-C34.9, C37-C38.8, C40-C41.9, C43-C45.9, C47-C54.9, C56-C57.8, C60-C63.8, C64-C67.9, C68.0-C68.8, C69.0-C69.8, C70-C73.9, C75-C75.8, C81-C82.9, C83.0-C83.8, C84-C85.0, C85.2-C85.8, C86-C86.6, C88-C91.0, C91.2-C91.3, C91.6, C92-C92.6, C93-C93.1, C93.3, C93.8, C94-C94.5, C94.7-C96.9, D00.1-D00.2, D01.0-D01.3, D02.0-D02.3, D03-D06.9, D07.0-D07.2, D07.4-D07.5, D09.0, D09.2-D09.3, D09.8, D10.0-D10.7, D11-D12.9, D13.0-D13.7, D14.0-D14.3, D15-D16.9, D22-D24.9, D26.0-D27.9, D28.0-D28.1, D28.7, D29.0-D29.8, D30.0-D30.8, D31-D36, D36.1-D36.7, D37.1-D37.5, D38.0-D38.5, D39.1-D39.2, D39.8, D40.0-D40.8, D41.0-D41.8, D42-D43.9, D44.0-D44.8, D45-D47.9, D48.0-D48.6, D49.2-D49.4, D49.6, K62.0-K62.1, K63.5, N60-N60.9, N84.0-N84.1, N87-N87.9",
    "Cardiovascular diseases": "B33.2, G45-G46.8, I01-I01.9, I02.0, I05-I09.9, I11-I11.9, I20-I25.9, I27.0, I27.2, I28-I28.9, I30-I31.1, I31.8-I37.8, I38-I41.9, I42.1-I42.8, I43-I43.9, I47-I48.9, I51.0-I51.4, I60-I63.9, I65-I66.9, I67.0-I67.3, I67.5-I67.6, I68.0-I68.2, I69.0-I69.3, I70.2-I70.8, I71-I73.9, I77-I83.9, I86-I89.0, I89.9, I98, K75.1",
    "Neglected tropical diseases and malaria": "A68-A68.9, A69.2-A69.9, A75-A75.9, A77-A79.9, A82-A82.9, A90-A96.9, A98-A98.8, B33.0-B33.1, B50-B53.8, B55.0, B56-B57.5, B60-B60.8, B65-B67.9, B69-B72.0, B74.3-B75, B77-B77.9, B83-B83.8, K93.1, P37.1, U06-U06.9",
    "Nutritional deficiencies": "D50.1-D50.8, D51-D52.0, D52.8-D53.9, E00-E02, E40-E46.9, E51-E61.9, E63-E64.0, E64.2-E64.9, M12.1",
    "Diabetes and kidney diseases": "D63.1, E10-E11.9, I12-I13.9, N00-N08.8, N15.0, N18-N18.9, P70.2, Q61-Q62.8",
    "Mental disorders": "F24, F50.0-F50.5",
    "Neurological disorders": "F00-F02.0, F02.2-F02.3, F02.8-F03.9, G10-G13.8, G20-G20.9, G23-G24, G24.1-G25.0, G25.2-G25.3, G25.5, G25.8-G26.0, G30-G31.1, G31.8-G31.9, G35-G37.9, G40-G41.9, G61-G61.9, G70-G71.1, G71.3-G72, G72.2-G73.7, G90-G90.9, G95-G95.9, M33-M33.9",
    "Self-harm and interpersonal violence": "U00-U03, X60-X64.9, X66-X83.9, X85-Y08.9, Y35-Y38.9, Y87.0-Y87.1, Y89.0-Y89.1",
    "COVID-19": "U07-U07.2",
    "Transport injuries": "V00-V86.9, V87.2-V87.3, V88.2-V88.3, V90-V98.8",
    "Digestive diseases": "B18-B18.9, I84-I85.9, I98.2, K20-K20.9, K22-K22.6, K22.8-K29.9, K31-K31.8, K35-K38.9, K40-K42.9, K44-K46.9, K50-K52, K52.8-K52.9, K55-K62, K62.2-K62.6, K62.8-K62.9, K64-K64.9, K66.8, K67, K68, K70-K70.3, K71.7, K73-K75, K75.2, K75.4-K76.2, K76.4-K77, K77.8, K80-K83.9, K85-K86.9, K90-K90.9, K92.8, K93.8, M09.1",
    "Unintentional injuries": "D52.1, D59.0, D59.2, D59.6, D69.5, D70.1-D70.2, D78-D78.8, E03.2, E06.4, E09-E09.9, E16.0, E23.1, E24.2, E27.3, E36-E36.8, E66.1, E88.3, E89-E89.9, G21.0-G21.1, G24.0, G25.1, G25.4, G25.6-G25.7, G72.0, G93.7, G97-G97.9, I95.2-I95.3, I97-I97.9, I98.9, J70.0-J70.5, J95-J95.9, K43-K43.9, K52.0, K62.7, K91-K91.9, K94-K95.8, L55-L55.9, L56.3, L56.8-L56.9, L58-L58.9, M87.1, N14-N14.4, N30.4, N65-N65.1, N99-N99.9, P93-P93.8, P96.2, P96.5, R50.2, W00-W46.2, W49-W62.9, W64-W70.9, W73-W75.9, W77-W81.9, W83-W94.9, W97.9, W99-X06.9, X08-X39.9, X47-X48.9, X50-X54.9, X57-X58.9, Y40-Y84.9, Y88-Y88.3",
    "Diarrheal diseases": "A00-A00.9, A02-A02.0, A02.8-A07, A07.2-A07.4, A08-A09.9, K52.1-K52.3, R19.7",
    "HIV/AIDS and sexually transmitted infections": "A50-A58, A60-A60.9, A63-A63.8, B20-B24.9, B63, F02.4, I98.0, K67.0-K67.2, M03.1, M73.0-M73.1",
    "Respiratory infections and tuberculosis": "A10-A14, A15-A19.9, A48.1, A70, B34.2, B90-B90.9, B97.2, B97.4-B97.6, H70-H70.9, J00-J02.8, J03-J03.8, J04-J04.2, J05-J05.1, J06.0-J06.8, J09-J15.8, J16-J16.9, J20-J21.9, J36-J36.0, J91.0, K67.3, K93.0, M49.0, N74.1, P23.0-P23.4, P37.0, U04-U04.9, U07-U07.2, U84.3",
    "Chronic respiratory diseases": "D86-D86.2, D86.9, G47.3, J30-J35.9, J37-J39.9, J41-J46.9, J60-J63.8, J66-J68.9, J70, J70.8-J70.9, J82, J84-J84.9, J91, J91.8-J92.9",
}


def _build_cie10_gbd_map() -> dict:
    """CIE-10 de 3 chars -> nombre GBD Nivel 2 (solo dueño único; COVID override)."""
    import re
    from collections import defaultdict

    def to3(tok):
        m = re.match(r"^([A-Z])(\d{2})", tok.strip())
        return (m.group(1), int(m.group(2))) if m else None

    def expand(icd):
        out = set()
        for tok in icd.split(","):
            tok = tok.strip()
            if "-" in tok:
                a, b = tok.split("-", 1); s, e = to3(a), to3(b)
            else:
                s = e = to3(tok)
            if not s or not e:
                continue
            o1, o2 = (ord(s[0]) - 65) * 100 + s[1], (ord(e[0]) - 65) * 100 + e[1]
            for o in range(o1, o2 + 1):
                n = o % 100
                if n <= 99:
                    out.add(f"{chr(65 + o // 100)}{n:02d}")
        return out

    owners = defaultdict(set)
    for name, icd in GBD_L2_RANGES.items():
        for c in expand(icd):
            owners[c].add(name)
    owners["U07"] = {"COVID-19"}
    # ── Overrides: códigos de 3 chars que el master GBD deja en gaps pero que
    # lógicamente pertenecen a una de las 16 causas (verificación manual vs WHO
    # Mortality). Son códigos muy comunes en Guatemala (J18 neumonía, E14
    # diabetes no insulinodependiente, I64 AVC, I50 insuficiencia cardíaca, etc.)
    # sin los cuales la cobertura GBD cae al 48%. Ver docs/modelo_dimensional.md
    # §6.4 "brecha conocida".
    _CIE3_OVERRIDES_16 = {
        "I10": "Cardiovascular diseases",   # Hipertensiva (I11 está, I10 no)
        "I50": "Cardiovascular diseases",   # Insuficiencia cardíaca (gap I48.9-I51.0)
        "I64": "Cardiovascular diseases",   # AVC no especificado (gap I63.9-I65)
        "I26": "Cardiovascular diseases",   # Embolia pulmonar (gap I25.9-I27.0)
        "I99": "Cardiovascular diseases",   # Trastornos circulatorios residuales
        "E14": "Diabetes and kidney diseases",  # Diabetes no insulinodep. (solo E10-E11)
        "N17": "Diabetes and kidney diseases",  # AKI (solo N18 está)
        "N19": "Diabetes and kidney diseases",  # Insuficiencia renal (solo N18)
        "N39": "Diabetes and kidney diseases",  # Trastornos urinarios
        "J18": "Respiratory infections and tuberculosis",  # Neumonía (gap J15.8-J16)
        "J96": "Respiratory infections and tuberculosis",  # Insuficiencia respiratoria
        "V89": "Transport injuries",            # Accidente vehículo no especificado
        "X45": "Unintentional injuries",        # Envenenamiento accidental (gap X39.9-X47)
        "X49": "Unintentional injuries",        # Envenenamiento accidental (gap X48.9-X50)
        "X59": "Unintentional injuries",        # Exposición no especificada (gap X54.9-X57)
        "W76": "Unintentional injuries",        # Ahogamiento accidental (gap W74.9-W77)
        "K72": "Digestive diseases",            # Insuficiencia hepática (gap K71.7-K73)
        "K65": "Digestive diseases",            # Peritonitis (gap K64.9-K66.8)
        "Y09": "Self-harm and interpersonal violence",  # Agresión (gap Y08.9-Y35)
        "C74": "Neoplasms",                     # Tumor suprarrenal (gap C73.9-C75)
        "C80": "Neoplasms",                     # Tumor maligno no especificado
        "C55": "Neoplasms",                     # Tumor maligno del útero (gap C54.9-C56)
    }
    for c, name in _CIE3_OVERRIDES_16.items():
        owners[c] = {name}
    return {c: next(iter(s)) for c, s in owners.items() if len(s) == 1}


# Códigos de 3 chars que pertenecen a causas GBD Nivel 2 FUERA de las 16. Se
# puebla SOLO gbd_nombre (autoritativo); gbd_code/gbd_cause_id quedan NULL
# porque el CGxxxx no existe en ninguna fuente confiable (no se fabrica). Ver
# docs/modelo_dimensional.md §6.4 (misma regla que Panamá lista-103).
_CIE3_NOMBRES_FUERA_16 = {
    "A41": "Other infectious diseases",     # Sepsis
    "F10": "Substance use disorders",       # Trastornos por alcohol
    "P07": "Neonatal disorders", "P21": "Neonatal disorders",
    "P22": "Neonatal disorders", "P24": "Neonatal disorders",
    "P36": "Neonatal disorders",
    "Q24": "Congenital birth defects", "Q89": "Congenital birth defects",
    "Q90": "Congenital birth defects",
    "D64": "Blood disorders", "D65": "Blood disorders",
    "E86": "Nutritional deficiencies",      # Desnutrición calórica
    "E87": "Nutritional deficiencies",      # Trastornos de líquidos/electrolitos
}
_NOMBRES_FUERA16_MAP = F.create_map(
    *[x for k, v in _CIE3_NOMBRES_FUERA_16.items() for x in (F.lit(k), F.lit(v))]
)


# Filas de referencia (3char, gbd_code, gbd_nombre, gbd_cause_id) para el join.
_CIE10_GBD_ROWS = [
    (c, MAPEO_GBD[n][0], n, MAPEO_GBD[n][1])
    for c, n in _build_cie10_gbd_map().items()
]

_SENTINEL_COLS = [
    "dep_registro", "mun_registro", "mes_registro",
    "dep_ocurrencia", "mun_ocurrencia", "mes_ocurrencia",
    "sexo", "periodo_edad",
    "estado_civil", "escolaridad", "nacionalidad",
    "dep_residencia", "mun_residencia", "lugar_residencia",
    "lugar_defuncion", "asistencia_medica", "tipo_ocurrencia",
]

_INE_COLUMN_RENAMES = {
    "Depreg":  "dep_registro",
    "Mupreg":  "mun_registro",
    "Mesreg":  "mes_registro",
    "Añoreg":  "anio_registro",
    "Depocu":  "dep_ocurrencia",
    "Mupocu":  "mun_ocurrencia",
    "Sexo":    "sexo",
    "Diaocu":  "dia_ocurrencia",
    "Mesocu":  "mes_ocurrencia",
    "Añoocu":  "anio_ocurrencia",
    "Edadif":  "edad",
    "Perdif":  "periodo_edad",
    "Ecidif":  "estado_civil",
    "Escodif": "escolaridad",
    "Ciuodif": "lugar_defuncion",
    "Nacdif":  "nacionalidad",
    "Predif":  "dep_residencia",
    "Dredif":  "mun_residencia",
    "Mredif":  "lugar_residencia",
    "Caudef":  "causa_cie10",
    "Asist":   "asistencia_medica",
    "Ocur":    "tipo_ocurrencia",
}

_INE_DROP_COLS = ["Areag", "Puedif", "Pnadif", "Dnadif", "Mnadif", "Cerdef", "Mredof"]

_MESES = {
    "Enero": 1, "Febrero": 2, "Marzo": 3, "Abril": 4,
    "Mayo": 5, "Junio": 6, "Julio": 7, "Agosto": 8,
    "Septiembre": 9, "Octubre": 10, "Noviembre": 11, "Diciembre": 12,
}

# Columna del stage -> nombre de variable en el diccionario INE.
# Los archivos viejos (2015-2017) traen NOMBRES; los nuevos traen CÓDIGOS.
# La decodificación deja todo en nombres para que stage.ine sea consistente.
# Nota: algunos nombres de columna no coinciden semánticamente con su contenido
# real (herencia del esquema original), por eso el catálogo se asigna por el
# significado REAL de la columna de origen:
#   dep_residencia  <- Predif = País de residencia
#   mun_residencia  <- Dredif = Departamento de residencia
#   lugar_residencia<- Mredif = Municipio de residencia
#   lugar_defuncion <- Ciuodif = Ocupación (CIUO-08)
#   tipo_ocurrencia <- Ocur = Sitio de ocurrencia
_COLUMN_CATALOG = {
    "dep_registro":      "Departamento de registro",
    "mun_registro":      "Municipio de registro",
    "mes_registro":      "Mes de registro",
    "dep_ocurrencia":    "Departamento de ocurrencia",
    "mun_ocurrencia":    "Municipio de ocurrencia",
    "sexo":              "Sexo del difunto(a)",
    "mes_ocurrencia":    "Mes de ocurrencia",
    "periodo_edad":      "Periodo de edad del difunto(a)",
    "estado_civil":      "Estado civil del difunto(a)",
    "escolaridad":       "Escolaridad del difunto(a)",
    "lugar_defuncion":   "Ocupación (Subgrupos CIUO-08) del difunto(a)",
    "nacionalidad":      "Nacionalidad del difunto(a)",
    "dep_residencia":    "Pais de residencia del difunto(a)",
    "mun_residencia":    "Departamento de residencia del difunto(a)",
    "lugar_residencia":  "Municipio de residencia del difunto(a)",
    "asistencia_medica": "Asistencia recibida",
    "tipo_ocurrencia":   "Sitio de ocurrencia",
}

EXPECTED_SCHEMA = {
    "id":                 LongType(),
    "pais_iso3":          StringType(),
    "dep_registro":       StringType(),
    "mun_registro":       StringType(),
    "mes_registro":       StringType(),
    "anio_registro":      IntegerType(),
    "dep_ocurrencia":     StringType(),
    "mun_ocurrencia":     StringType(),
    "dia_ocurrencia":     IntegerType(),
    "mes_ocurrencia":     StringType(),
    "mes_ocurrencia_num": IntegerType(),
    "anio_ocurrencia":    IntegerType(),
    "fecha_ocurrencia":   DateType(),
    "sexo":               StringType(),
    "es_masculino":       BooleanType(),
    "es_femenino":        BooleanType(),
    "es_total":           BooleanType(),
    "es_desconocido":     BooleanType(),
    "edad":               IntegerType(),
    "periodo_edad":       StringType(),
    "es_menor_de_1_anio": BooleanType(),
    "id_etario":          IntegerType(),
    "categoria_etaria":   StringType(),
    "estado_civil":       StringType(),
    "escolaridad":        StringType(),
    "nacionalidad":       StringType(),
    "dep_residencia":     StringType(),
    "mun_residencia":     StringType(),
    "lugar_residencia":   StringType(),
    "causa_cie10":        StringType(),
    "cie10_code":         StringType(),
    "cie10_nombre":       StringType(),
    "gbd_code":           StringType(),
    "gbd_nombre":         StringType(),
    "gbd_cause_id":       IntegerType(),
    "lugar_defuncion":    StringType(),
    "asistencia_medica":  StringType(),
    "tipo_ocurrencia":    StringType(),
    "anio":               IntegerType(),
    "source_file":        StringType(),
    "ingestion_ts":       TimestampType(),
}


def _verify_schema(df: DataFrame, expected: dict, table_name: str) -> None:
    actual = {f.name: f.dataType for f in df.schema.fields}
    errors = []
    for col, exp_type in expected.items():
        if col not in actual:
            errors.append(f"  MISSING  '{col}'")
        elif type(actual[col]) != type(exp_type):
            errors.append(f"  MISMATCH '{col}': expected {exp_type}, got {actual[col]}")
    extra = [c for c in actual if c not in expected]
    if extra:
        errors.append(f"  EXTRA    {extra}")
    if errors:
        print(f"[SCHEMA] {table_name} — {len(errors)} problema(s):")
        for msg in errors:
            print(msg)
    else:
        print(f"[SCHEMA] {table_name} — OK ({len(actual)} columnas, tipos correctos)")


def _rename_ine_columns(df: DataFrame) -> DataFrame:
    for src, dst in _INE_COLUMN_RENAMES.items():
        if src in df.columns:
            df = df.withColumnRenamed(src, dst)
    return df


def _drop_unused_columns(df: DataFrame) -> DataFrame:
    cols_to_drop = [c for c in _INE_DROP_COLS if c in df.columns]
    return df.drop(*cols_to_drop) if cols_to_drop else df


def _replace_ignorado(df: DataFrame) -> DataFrame:
    # 'Ignorado' aparece como marcador en columnas numéricas (edad, dia_ocurrencia,
    # años). Debe anularse ANTES de cualquier cast: con ANSI mode activo, castear
    # o comparar 'Ignorado' contra un entero lanza CAST_INVALID_INPUT.
    return df.replace("Ignorado", None)


def _clean_text(col):
    # Normaliza espacios no separables ( ) y BOM (﻿), colapsa espacios
    # y recorta. Los CSV del INE (codificación Windows) suelen traer estos
    # caracteres invisibles; sin limpiarlos, las columnas validadas contra
    # catálogos fijos (área, mes) no hacen match y quedan en null.
    cleaned = F.regexp_replace(col, "[ ﻿]", " ")
    return F.trim(F.regexp_replace(cleaned, r"\s+", " "))


def _add_mes_ocurrencia_num(df: DataFrame) -> DataFrame:
    # Soporta el mes como nombre ("Agosto", insensible a may/min y a espacios
    # invisibles) y como código numérico ("8"). create_map con varargs por
    # compatibilidad con Spark Connect.
    mes = _clean_text(F.col("mes_ocurrencia"))
    name_map = F.create_map(
        *[item for k, v in _MESES.items() for item in (F.lit(k.lower()), F.lit(v))]
    )
    by_name = name_map[F.lower(mes)]
    by_num = F.when(mes.rlike(r"^(0?[1-9]|1[0-2])$"), mes.cast(IntegerType()))
    return df.withColumn(
        "mes_ocurrencia_num",
        F.coalesce(by_name, by_num).cast(IntegerType()),
    )


def _trim_geo(df: DataFrame) -> DataFrame:
    for col in ("dep_registro", "mun_registro", "dep_ocurrencia",
                "mun_ocurrencia", "dep_residencia", "mun_residencia"):
        df = df.withColumn(col, F.trim(F.col(col)))
    return df



def _cast_year(df: DataFrame, col: str, lo: int = 1900, hi: int = 2030) -> DataFrame:
    return df.withColumn(col,
        F.when(F.col(col).cast(IntegerType()).between(lo, hi),
               F.col(col).cast(IntegerType()))
         .otherwise(F.lit(None).cast(IntegerType()))
    )


def _cast_day(df: DataFrame) -> DataFrame:
    return df.withColumn("dia_ocurrencia",
        F.when(F.col("dia_ocurrencia").cast(IntegerType()).between(1, 31),
               F.col("dia_ocurrencia").cast(IntegerType()))
         .otherwise(F.lit(None).cast(IntegerType()))
    )


def _build_fecha_ocurrencia(df: DataFrame) -> DataFrame:
    return df.withColumn("fecha_ocurrencia",
        F.when(
            F.col("dia_ocurrencia").isNotNull()
            & F.col("mes_ocurrencia_num").isNotNull()
            & F.col("anio_ocurrencia").isNotNull(),
            F.to_date(
                F.concat_ws("-",
                    F.col("anio_ocurrencia").cast("string"),
                    F.lpad(F.col("mes_ocurrencia_num").cast("string"), 2, "0"),
                    F.lpad(F.col("dia_ocurrencia").cast("string"), 2, "0"),
                ),
                "yyyy-MM-dd",
            )
        ).otherwise(F.lit(None).cast(DateType()))
    )


def _add_sex_booleans(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("_sx", F.lower(F.trim(F.col("sexo"))))
        .withColumn("es_masculino",
            F.when(F.col("_sx") == "hombre", F.lit(True)).otherwise(F.lit(False)))
        .withColumn("es_femenino",
            F.when(F.col("_sx") == "mujer",  F.lit(True)).otherwise(F.lit(False)))
        .withColumn("es_total",       F.lit(False).cast(BooleanType()))
        .withColumn("es_desconocido",
            F.when(F.col("_sx").isin("hombre", "mujer"), F.lit(False))
             .otherwise(F.lit(True)))
        .drop("_sx")
    )


def _load_catalogs() -> dict:
    # Construye {variable: {codigo: etiqueta}} desde el diccionario INE ingerido.
    # El CSV trae la variable solo en la primera fila de cada bloque (forward-fill).
    # Bajo Spark Connect, spark.read.table() es perezoso y no resuelve la tabla
    # hasta la acción (.collect()), así que un try/except alrededor de read.table
    # no atrapa TABLE_OR_VIEW_NOT_FOUND. Verificamos la existencia explícitamente.
    if not spark.catalog.tableExists("semi2.sandbox.raw_ine_diccionario"):
        print("[WARN] sandbox.raw_ine_diccionario no existe; se omite la decodificación de códigos.")
        return {}
    d = spark.read.table("semi2.sandbox.raw_ine_diccionario")
    catalogs: dict = {}
    current = None
    for row in d.select("_c0", "_c1", "_c2").collect():
        valor    = (row[0] or "").strip()
        codigo   = (row[1] or "").strip()
        etiqueta = (row[2] or "").strip()
        if valor in ("Valores de las variables defunciones", "Valor"):
            continue
        if codigo == "Código":
            continue
        if valor:
            current = valor
            catalogs.setdefault(current, {})
        if current and codigo and etiqueta:
            catalogs[current][codigo] = etiqueta
    return catalogs


def _decode_coded_columns(df: DataFrame, catalogs: dict) -> DataFrame:
    # Reemplaza códigos por etiquetas usando el catálogo de cada columna.
    # Los valores que ya son nombres (años viejos) no están en el catálogo y
    # pasan sin cambios vía coalesce.
    for col, varname in _COLUMN_CATALOG.items():
        cat = catalogs.get(varname)
        if not cat or col not in df.columns:
            continue
        mapping = F.create_map(
            *[item for k, v in cat.items() for item in (F.lit(k), F.lit(v))]
        )
        cleaned = _clean_text(F.col(col))
        df = df.withColumn(col, F.coalesce(mapping[cleaned], cleaned))
    return df


def _validate_edad(df: DataFrame) -> DataFrame:
    # Réplica de la lógica original: edad en meses (periodo_edad contiene "mes")
    # cuenta como 0 años; fuera de [0, 120] → null; el resto se castea a entero.
    edad_int = F.col("edad").cast(IntegerType())
    return df.withColumn("edad",
        F.when(F.col("periodo_edad").contains("mes"), F.lit(0))
         .when(edad_int.between(0, 120), edad_int)
         .otherwise(F.lit(None).cast(IntegerType()))
    )


def _validate_cie10(df: DataFrame) -> DataFrame:
    return df.withColumn("causa_cie10",
        F.when(
            F.col("causa_cie10").isNotNull()
            & F.upper(F.trim(F.col("causa_cie10"))).rlike(CIE10_RE),
            F.upper(F.trim(F.col("causa_cie10")))
        ).otherwise(F.lit(None).cast(StringType()))
    )


def _apply_sentinels(df: DataFrame, cols: list, sentinels: tuple) -> DataFrame:
    for c in cols:
        df = df.withColumn(c, F.trim(F.col(c)))
        df = df.withColumn(c,
            F.when(F.col(c).isin(*sentinels), F.lit(None).cast(StringType()))
             .otherwise(F.col(c))
        )
    return df


def _add_menor_de_1_anio(df: DataFrame) -> DataFrame:
    return df.withColumn("es_menor_de_1_anio",
        F.when(F.col("periodo_edad").isNull(),          F.lit(None).cast(BooleanType()))
         .when(F.col("periodo_edad").contains("mes"),   F.lit(True))
         .otherwise(                                    F.lit(False))
    )


def _add_etario_columns(df: DataFrame) -> DataFrame:
    # INE trae año simple `edad` (0-120); periodo_edad distingue <1 año
    # ('Menos de un mes', '1 a 11 meses' -> ya está casteado edad=0 en
    # _validate_edad). Crosswalk §5.3: edad -> banda del grid canónico.
    edad = F.col("edad").cast(IntegerType())
    return (
        df
        .withColumn("id_etario",
            F.when(F.col("edad").isNull(),                              F.lit(98))   # UNK
             .when(edad == 0,                                           F.lit(1))    # LT1
             .when(edad.between(1, 4),                                  F.lit(2))    # 01-04
             .when(edad.between(5, 9),                                  F.lit(3))    # 05-09
             .when(edad.between(10, 14),                                F.lit(4))    # 10-14
             .when(edad.between(15, 19),                                F.lit(5))    # 15-19
             .when(edad.between(20, 24),                                F.lit(6))    # 20-24
             .when(edad.between(25, 29),                                F.lit(7))    # 25-29
             .when(edad.between(30, 34),                                F.lit(8))    # 30-34
             .when(edad.between(35, 39),                                F.lit(9))    # 35-39
             .when(edad.between(40, 44),                                F.lit(10))   # 40-44
             .when(edad.between(45, 49),                                F.lit(11))   # 45-49
             .when(edad.between(50, 54),                                F.lit(12))   # 50-54
             .when(edad.between(55, 59),                                F.lit(13))   # 55-59
             .when(edad.between(60, 64),                                F.lit(14))   # 60-64
             .when(edad.between(65, 69),                                F.lit(15))   # 65-69
             .when(edad.between(70, 74),                                F.lit(16))   # 70-74
             .when(edad.between(75, 79),                                F.lit(17))   # 75-79
             .when(edad.between(80, 84),                                F.lit(18))   # 80-84
             .when(edad >= 85,                                          F.lit(19))   # 85+
             .otherwise(F.lit(None).cast(IntegerType()))
        )
        .withColumn("categoria_etaria", _ETARIO_CAT_MAP[F.col("id_etario")])
    )


def _load_cie10_nombres() -> dict:
    """CIE-10 (código de 3-4 chars) -> descripción en español desde el diccionario INE.

    El CSV se ingesta sin header (_c0=código, _c1=descripción); las 2 primeras
    filas son metadatos ('Lista de categorías...', 'Código CIE-10,Descripción')
    y se filtran con el patrón CIE-10. Si la tabla no existe, devuelve {} y
    cie10_nombre queda NULL (best-effort, §6.3).
    """
    import re
    if not spark.catalog.tableExists("semi2.sandbox.raw_ine_diccionario_cie10"):
        print("[WARN] sandbox.raw_ine_diccionario_cie10 no existe; cie10_nombre quedará NULL.")
        return {}
    rows = (
        spark.read.table("semi2.sandbox.raw_ine_diccionario_cie10")
             .select("_c0", "_c1").collect()
    )
    cat = {}
    for row in rows:
        code = (row[0] or "").strip()
        desc = (row[1] or "").strip()
        if re.match(r"^[A-Z][0-9]{2}[0-9A-Z]?$", code) and desc:
            cat[code] = desc
    return cat


def _add_gbd_columns(df: DataFrame, cie10_nombres: dict) -> DataFrame:
    # causa_cie10 (3-4 chars ya validados) -> padre GBD Nivel 2 por sus 3 primeros
    # chars, vía broadcast join contra la referencia (811 + overrides). cie10_nombre
    # se puebla desde el diccionario CIE-10 del INE (código -> descripción); si no
    # está disponible, queda NULL (best-effort, §6.3).
    # Para códigos fuera de las 16 causas (sepsis, neonatal, congénitas, etc.),
    # gbd_code/gbd_cause_id quedan NULL pero gbd_nombre se puebla desde
    # _NOMBRES_FUERA16_MAP (§6.4, misma regla que Panamá lista-103).
    ref = F.broadcast(
        spark.createDataFrame(_CIE10_GBD_ROWS, ["_cie3", "gbd_code", "gbd_nombre", "gbd_cause_id"])
    )
    if cie10_nombres:
        cie10_map = F.create_map(
            *[item for k, v in cie10_nombres.items() for item in (F.lit(k), F.lit(v))]
        )
        cie10_nombre_expr = cie10_map[F.col("causa_cie10")]
    else:
        cie10_nombre_expr = F.lit(None).cast(StringType())
    df = (
        df
        .withColumn("cie10_code",   F.col("causa_cie10"))
        .withColumn("cie10_nombre", cie10_nombre_expr)
        .withColumn("_cie3",        F.substring(F.col("causa_cie10"), 1, 3))
    )
    df = df.join(ref, on="_cie3", how="left").drop("_cie3")
    # gbd_nombre: si el join no encontró (código fuera de las 16), se intenta
    # desde _NOMBRES_FUERA16_MAP; si tampoco, queda NULL.
    df = df.withColumn("gbd_nombre",
        F.coalesce(F.col("gbd_nombre"), _NOMBRES_FUERA16_MAP[F.col("cie10_code").substr(1, 3)])
    )
    return df.withColumn("gbd_cause_id", F.col("gbd_cause_id").cast(IntegerType()))


def transform(df: DataFrame, catalogs: dict, cie10_nombres: dict) -> DataFrame:
    df = _rename_ine_columns(df)
    df = _drop_unused_columns(df)
    df = _replace_ignorado(df)
    # Decodifica códigos -> nombres ANTES de derivar (mes_num, fecha, booleanos
    # de sexo, edad por periodo) para que esa lógica vea siempre nombres.
    df = _decode_coded_columns(df, catalogs)
    df = _add_mes_ocurrencia_num(df)
    df = _trim_geo(df)
    df = _cast_day(df)
    df = _cast_year(df, "anio_ocurrencia")
    df = _cast_year(df, "anio_registro")
    df = _cast_year(df, "anio")
    df = _build_fecha_ocurrencia(df)
    df = _add_sex_booleans(df)
    df = _validate_edad(df)
    df = _validate_cie10(df)
    df = _add_gbd_columns(df, cie10_nombres)
    df = df.dropDuplicates()
    df = (
        df
        .withColumn("pais_iso3", F.lit(PAIS_ISO3))
        .withColumn("id", F.monotonically_increasing_id().cast(LongType()))
        .withColumnRenamed("_source_file",         "source_file")
        .withColumnRenamed("_ingestion_timestamp", "ingestion_ts")
    )
    df = _apply_sentinels(df, _SENTINEL_COLS, _SENTINELS)
    df = _add_menor_de_1_anio(df)
    df = _add_etario_columns(df)
    return df.select(
        "id", "pais_iso3",
        "dep_registro", "mun_registro", "mes_registro", "anio_registro",
        "dep_ocurrencia", "mun_ocurrencia",
        "dia_ocurrencia", "mes_ocurrencia", "mes_ocurrencia_num",
        "anio_ocurrencia", "fecha_ocurrencia",
        "sexo", "es_masculino", "es_femenino", "es_total", "es_desconocido",
        "edad", "periodo_edad", "es_menor_de_1_anio",
        "id_etario", "categoria_etaria",
        "estado_civil", "escolaridad", "nacionalidad",
        "dep_residencia", "mun_residencia", "lugar_residencia",
        "causa_cie10", "cie10_code", "cie10_nombre",
        "gbd_code", "gbd_nombre", "gbd_cause_id",
        "lugar_defuncion", "asistencia_medica", "tipo_ocurrencia",
        "anio", "source_file", "ingestion_ts",
    )


spark.sql("CREATE SCHEMA IF NOT EXISTS stage")

try:
    catalogs       = _load_catalogs()
    cie10_nombres  = _load_cie10_nombres()
    raw      = spark.read.table("semi2.sandbox.raw_ine")
    staged   = transform(raw, catalogs, cie10_nombres)
    _verify_schema(staged, EXPECTED_SCHEMA, "stage.ine")
    display(staged)
    staged.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("stage.ine")
except Exception as e:
    print(f"Error al procesar sandbox.raw_ine: {e}")
    raise
